# Homework 3 — Training pix2pixHD on BBBC010

We use NVIDIA's [pix2pixHD](https://github.com/NVIDIA/pix2pixHD) on the BBBC010 data. The dataset has 80 train pairs (mask → brightfield) at
512×512. We train for 40 epochs and save **milestone checkpoints** at epochs 5, 10, 20, 40
so the evaluation notebook can compare quality over time.

## 1. Clone the original pix2pixHD repo

In [ ]:
!git clone https://github.com/NVIDIA/pix2pixHD.git

## 2. Apply the Python-3.11+ patches

The original repo targeted Python 3.5 / PyTorch 0.4. `apply_patches.py` (sitting next to
`pix2pixHD/`) makes three small changes

In [ ]:
!python apply_patches.py

## 3. Inspect data layout

After running `00_PrepData.ipynb` you should have:
```
datasets/bbbc010_pix2pixhd/
├── train_A/   # 80 binary masks (RGB, 512×512)
├── train_B/   # 80 brightfield images
├── test_A/    # 20 masks
└── test_B/    # 20 images
```
Because we use `--label_nc 0`, pix2pixHD treats the input as a raw image, not a semantic map.

## 4. Training command

Same flags as Unit 5. **Milestone checkpoints** (epochs 5/10/20/40) come from
`apply_patches.py` — they let the evaluation notebook show how generation quality evolves.

Expect ~1–2 hours on a single modern GPU. Adjust `--batchSize` to fit your memory.

In [ ]:
# TODO: fill in the training hyperparameters
# Hints:
#   --loadSize / --fineSize: 512 (matches prep)
#   --batchSize: 2 (fits a single modern GPU at 512x512; lower to 1 if OOM)
#   --niter: 40 epochs, --niter_decay: 0 (no LR decay phase)
#   --save_epoch_freq: 100 (so only the milestone checkpoints from
#       apply_patches.py at epochs 5/10/20/40 get saved)
!cd pix2pixHD && python train.py \
    --name bbbc010_512 \
    --dataroot ../datasets/bbbc010_pix2pixhd \
    --label_nc 0 \
    --no_instance \
    --loadSize ___ \
    --fineSize ___ \
    --batchSize ___ \
    --niter ___ \
    --niter_decay ___ \
    --save_epoch_freq ___ \
    --gpu_ids 0 \
    --checkpoints_dir ./checkpoints

## 5. Verify checkpoints

After training, you should see `5_net_*.pth`, `10_net_*.pth`, `20_net_*.pth`,
`40_net_*.pth` plus `latest_net_*.pth` in `pix2pixHD/checkpoints/bbbc010_512/`.

In [ ]:
!ls -lthr pix2pixHD/checkpoints/bbbc010_512/

## Conclusion

Mask-conditioned brightfield image generation: the model has to fill in
plausible worm textures inside the mask outline while leaving the background empty.
Next: `02_Evaluation.ipynb` quantifies how well it learned this over the 40-epoch run.